# Sesion 10 - Series de tiempo e inferencia en streaming con Spark

**Producto:** aplicacion del modelo guardado en la sesion 09 sobre datos batch y, opcionalmente, sobre un stream Kafka.

## Proposito de la sesion

Continuar el flujo iniciado en la sesion 08 y formalizado en la sesion 09: usar la evidencia de observabilidad del pipeline Kafka + Spark, cargar el `PipelineModel` entrenado con Spark MLlib y generar predicciones de latencia sin volver a entrenar.

## Resultado esperado

Al finalizar, el estudiante debe tener predicciones de latencia generadas con el modelo guardado por la sesion 09, evidencia batch persistida en Parquet y una ruta opcional para ejecutar inferencia streaming sobre eventos Kafka.

La cadena completa queda asi:

```text
Sesion 08: Kafka + Spark observabilidad -> Parquet con latencyMs
Sesion 09: Parquet -> serie temporal -> entrenamiento MLlib -> modelo guardado
Sesion 10: modelo guardado -> inferencia batch y streaming opcional
```

## 1.1 Teoria usada en esta practica

La teoria de entrenamiento, features, label y Pipeline queda en la sesion 09. En esta sesion usamos solo la teoria necesaria para inferencia:

- **Inferencia**: aplicar un modelo ya entrenado sobre datos nuevos. No se recalculan coeficientes ni se vuelve a ajustar el modelo.
- **Contrato de features**: el modelo espera las mismas columnas usadas en la sesion 09. Si falta una columna o cambia su significado, la prediccion deja de ser confiable.
- **PipelineModel**: artefacto guardado que contiene tanto las transformaciones como el modelo final. Spark ejecuta transform() para agregar la columna prediction.
- **Inferencia batch**: lee datos historicos en Parquet, reconstruye ventanas y genera predicciones reproducibles.
- **Inferencia streaming**: lee eventos nuevos desde Kafka, agrega por ventanas y aplica el mismo modelo dentro de micro-batches.
- **Evaluacion offline**: en batch podemos construir una etiqueta futura con lead() para comparar prediccion vs realidad. En streaming real, esa etiqueta aun no existe al momento de predecir.

Resumen de responsabilidades:

~~~text
Sesion 09: aprende el modelo
Sesion 10: reutiliza el modelo
~~~

## 1. Idea general

El flujo de inferencia mantiene tres reglas:

1. Cargar el mismo modelo guardado por la sesion 09 en `../artifacts/models/ml_latency_pipeline_lr`.
2. Reconstruir exactamente las mismas columnas de entrada usadas al entrenar: `eventCount`, `avgLatencyMs`, `minLatencyMs`, `maxLatencyMs`, `stdLatencyMs`, `latencyRangeMs`, `hourOfDay` y `minuteOfHour`.
3. Separar la inferencia batch de la inferencia streaming para observar como cambia la ejecucion.

La prediccion significa: **dada la ventana actual, estimar la latencia promedio de la siguiente ventana**.

## 2. Crear SparkSession

In [1]:
from pyspark.sql import SparkSession

KAFKA_CONNECTOR_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.2"

spark = (
    SparkSession.builder
    .appName("s10-series-tiempo-inferencia-spark")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.packages", KAFKA_CONNECTOR_PACKAGE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-faf9fb5d-2f08-499a-ac1c-141928fe0a97;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

## 3. Parametros de trabajo

In [2]:
from pathlib import Path
from pyspark.sql import functions as F

MODEL_PATHS = [
    "../artifacts/models/ml_latency_pipeline_lr",
    "/opt/artifacts/models/ml_latency_pipeline_lr",
]

BATCH_INPUT_PATHS = [
    "../artifacts/output/orden_eventos_observabilidad_p1",
    "../artifacts/output/orden_eventos_parquet",
    "/opt/artifacts/output/orden_eventos_observabilidad_p1",
    "/opt/artifacts/output/orden_eventos_parquet",
]

PREDICTIONS_BATCH_PATH = "../artifacts/output/ml_latency_predictions_batch"
PREDICTIONS_STREAM_PATH = "../artifacts/output/ml_latency_predictions_stream"
CHECKPOINT_STREAM_PATH = "../artifacts/checkpoint/ml-latency-inference-stream"

# En el stack Kafka actual de LambdaLab, el broker interno se llama kafka:9092.
# Si ejecutas este notebook dentro del contenedor PySpark, ese contenedor debe estar unido a lambdalab-kafka-net.
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
TOPIC_ORDENES = "orden-eventos"

## 4. Helpers: datos, serie temporal y features

Estas funciones son deliberadamente parecidas a la sesion 9. En inferencia, cambiar el contrato de features es una de las formas mas comunes de romper un modelo ya entrenado.

In [3]:
def first_existing_path(paths):
    for path in paths:
        if Path(path).exists():
            return path
    return None


def load_observability_batch():
    source_path = first_existing_path(BATCH_INPUT_PATHS)

    if source_path:
        df = spark.read.parquet(source_path)
        print(f"Fuente batch: {source_path}")
        return df

    print("No se encontraron Parquet reales. Se creara un dataset sintetico para la practica.")
    base = spark.range(0, 360)
    return base.select(
        (F.timestamp_seconds(F.lit(1746300000) + F.col("id") * 20)).alias("kafkaTimestamp"),
        (
            F.lit(120)
            + (F.col("id") % 12) * 8
            + F.when((F.col("id") % 41) == 0, 190).otherwise(0)
            + (F.rand(seed=99) * 35)
        ).cast("long").alias("latencyMs"),
        F.lit("ordenes").alias("topic"),
        (F.col("id") % 3).cast("int").alias("partition"),
        F.col("id").cast("long").alias("offset"),
        F.lit("demo").alias("origen"),
        F.lit(True).alias("isValid")
    )


def build_latency_series(input_df, ordered=True):
    result = input_df \
        .filter(F.col("kafkaTimestamp").isNotNull()) \
        .filter(F.col("latencyMs").isNotNull()) \
        .groupBy(F.window("kafkaTimestamp", "1 minute")) \
        .agg(
            F.count("*").alias("eventCount"),
            F.avg("latencyMs").alias("avgLatencyMs"),
            F.min("latencyMs").alias("minLatencyMs"),
            F.max("latencyMs").alias("maxLatencyMs"),
            F.stddev("latencyMs").alias("stdLatencyMs")
        ) \
        .withColumn("windowStart", F.col("window.start")) \
        .withColumn("windowEnd", F.col("window.end")) \
        .drop("window") \
        .na.fill({"stdLatencyMs": 0.0})

    if ordered:
        result = result.orderBy("windowStart")

    return result


def add_inference_features(series_df):
    return series_df \
        .withColumn("latencyRangeMs", F.col("maxLatencyMs") - F.col("minLatencyMs")) \
        .withColumn("hourOfDay", F.hour("windowStart")) \
        .withColumn("minuteOfHour", F.minute("windowStart"))

## 5. Cargar datos batch y reconstruir la serie

In [4]:
df_batch = load_observability_batch()

df_batch.printSchema()
df_batch.show(5, truncate=False)

serie_latency = build_latency_series(df_batch)
features_batch = add_inference_features(serie_latency)

features_batch.show(20, truncate=False)
print("Ventanas disponibles:", features_batch.count())

Fuente batch: ../artifacts/output/orden_eventos_observabilidad_p1
root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)

+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total |estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------

## 6. Cargar el modelo guardado

Esta sesion depende del modelo entrenado y guardado en la sesion 09. Si el modelo no existe, no entrenamos uno nuevo aqui: se debe volver a la sesion 09 y ejecutar hasta la celda `Guardar y cargar el modelo`.

Esto mantiene separadas las responsabilidades:

```text
Sesion 09 = entrenamiento
Sesion 10 = inferencia
```

In [5]:
from pyspark.ml import PipelineModel
from pyspark.sql.window import Window

feature_cols = [
    "eventCount",
    "avgLatencyMs",
    "minLatencyMs",
    "maxLatencyMs",
    "stdLatencyMs",
    "latencyRangeMs",
    "hourOfDay",
    "minuteOfHour",
]

model_path = first_existing_path(MODEL_PATHS)

if model_path is None:
    raise FileNotFoundError(
        "No se encontro el modelo de la sesion 09. "
        "Ejecuta 09_ml_distribuido_regresion_mllib.ipynb hasta guardar "
        "../artifacts/models/ml_latency_pipeline_lr."
    )

loaded_model = PipelineModel.load(model_path)
print(f"Modelo cargado desde: {model_path}")

Modelo cargado desde: ../artifacts/models/ml_latency_pipeline_lr


## 7. Inferencia batch

Aqui el modelo recibe ventanas historicas y produce una prediccion para cada una. Esta es la forma mas simple de validar que el artefacto guardado se puede reutilizar.

In [6]:
batch_predictions = loaded_model.transform(features_batch)

batch_predictions.select(
    "windowStart",
    "windowEnd",
    "eventCount",
    F.round("avgLatencyMs", 2).alias("avgLatencyMs"),
    F.round("prediction", 2).alias("predictedNextAvgLatencyMs")
).orderBy("windowStart").show(20, truncate=False)

+-------------------+-------------------+----------+------------+-------------------------+
|windowStart        |windowEnd          |eventCount|avgLatencyMs|predictedNextAvgLatencyMs|
+-------------------+-------------------+----------+------------+-------------------------+
|2026-05-04 09:25:00|2026-05-04 09:26:00|1         |50.0        |174.2                    |
|2026-05-04 09:26:00|2026-05-04 09:27:00|2         |554.0       |104.65                   |
|2026-05-04 09:27:00|2026-05-04 09:28:00|1         |235.0       |135.07                   |
|2026-05-04 10:48:00|2026-05-04 10:49:00|4         |1165.0      |-119.75                  |
+-------------------+-------------------+----------+------------+-------------------------+



26/06/08 03:38:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


## 8. Evaluacion batch con etiqueta futura

En inferencia productiva la etiqueta futura todavia no existe. Para evaluacion offline, usamos `lead(avgLatencyMs)` como etiqueta y comparamos contra la prediccion.

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

w = Window.orderBy("windowStart")

evaluation_df = batch_predictions \
    .withColumn("label", F.lead("avgLatencyMs", 1).over(w)) \
    .na.drop(subset=["label", "prediction"])

metrics = {}
for metric_name in ["rmse", "mae", "r2"]:
    evaluator = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metrics[metric_name] = evaluator.evaluate(evaluation_df)

for name, value in metrics.items():
    print(f"{name.upper()}: {value:.4f}")

evaluation_df \
    .withColumn("errorMs", F.col("label") - F.col("prediction")) \
    .withColumn("absErrorMs", F.abs(F.col("errorMs"))) \
    .select(
        "windowStart",
        F.round("avgLatencyMs", 2).alias("currentAvgLatencyMs"),
        F.round("label", 2).alias("realNextAvgLatencyMs"),
        F.round("prediction", 2).alias("predictedNextAvgLatencyMs"),
        F.round("absErrorMs", 2).alias("absErrorMs")
    ) \
    .orderBy(F.desc("absErrorMs")) \
    .show(20, truncate=False)

26/06/08 03:38:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 0

RMSE: 638.2238
MAE: 513.3603
R2: -1.7358


26/06/08 03:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 03:38:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/08 0

+-------------------+-------------------+--------------------+-------------------------+----------+
|windowStart        |currentAvgLatencyMs|realNextAvgLatencyMs|predictedNextAvgLatencyMs|absErrorMs|
+-------------------+-------------------+--------------------+-------------------------+----------+
|2026-05-04 09:27:00|235.0              |1165.0              |135.07                   |1029.93   |
|2026-05-04 09:25:00|50.0               |554.0               |174.2                    |379.8     |
|2026-05-04 09:26:00|554.0              |235.0               |104.65                   |130.35    |
+-------------------+-------------------+--------------------+-------------------------+----------+



## 9. Guardar predicciones batch

Esto deja evidencia del producto de la sesion: el modelo fue aplicado y sus resultados quedaron como Parquet.

In [8]:
batch_output = batch_predictions.select(
    "windowStart",
    "windowEnd",
    "eventCount",
    "avgLatencyMs",
    "minLatencyMs",
    "maxLatencyMs",
    "stdLatencyMs",
    "latencyRangeMs",
    "hourOfDay",
    "minuteOfHour",
    F.col("prediction").alias("predictedNextAvgLatencyMs")
)

batch_output.write.mode("overwrite").parquet(PREDICTIONS_BATCH_PATH)

print(f"Predicciones batch guardadas en: {PREDICTIONS_BATCH_PATH}")
spark.read.parquet(PREDICTIONS_BATCH_PATH).orderBy("windowStart").show(10, truncate=False)

Predicciones batch guardadas en: ../artifacts/output/ml_latency_predictions_batch
+-------------------+-------------------+----------+------------+------------+------------+------------------+--------------+---------+------------+-------------------------+
|windowStart        |windowEnd          |eventCount|avgLatencyMs|minLatencyMs|maxLatencyMs|stdLatencyMs      |latencyRangeMs|hourOfDay|minuteOfHour|predictedNextAvgLatencyMs|
+-------------------+-------------------+----------+------------+------------+------------+------------------+--------------+---------+------------+-------------------------+
|2026-05-04 09:25:00|2026-05-04 09:26:00|1         |50.0        |50          |50          |0.0               |0             |9        |25          |174.19590114385645       |
|2026-05-04 09:26:00|2026-05-04 09:27:00|2         |554.0       |322         |786         |328.09754647055803|464           |9        |26          |104.64856121664866       |
|2026-05-04 09:27:00|2026-05-04 09:28:00|1 

## 10.1 Teoria: inferencia streaming en micro-batches

Spark Structured Streaming procesa el flujo como una secuencia de micro-batches. Cada micro-batch contiene eventos disponibles en un intervalo corto y se transforma con operaciones de DataFrame.

En esta practica:

- Kafka entrega eventos de orden-eventos.
- Spark calcula latencyMs y agrega por ventanas de tiempo.
- El PipelineModel aplica transform() sobre cada conjunto de ventanas.
- foreachBatch permite tratar cada micro-batch como un DataFrame batch para guardar predicciones en Parquet.

La diferencia importante es que el modelo no cambia. Cambia la fuente de datos: historica en batch, viva en Kafka streaming.

## 10. Preparar lectura Kafka streaming opcional

Ejecuta esta parte solo si quieres aplicar el modelo sobre eventos nuevos que llegan por Kafka. La inferencia batch de los pasos anteriores es el flujo principal de la sesion.

Para que Spark pueda leer Kafka se necesitan dos cosas: el contenedor PySpark debe estar en la red de Kafka y la SparkSession debe declarar el paquete spark-sql-kafka. El paquete ya esta declarado al inicio del notebook.

Desde la raiz de lambdalab:

~~~powershell
docker compose -f kafka/compose.yml up -d
docker compose -f pyspark/compose.yml -f pyspark/compose.kafka.yml up -d --build
~~~

El override conecta el contenedor a lambdalab-kafka-net. Si abriste Jupyter solo con pyspark/compose.yml, reinicia el contenedor con el comando anterior y vuelve a crear la SparkSession antes de ejecutar esta parte.

El topic esperado es orden-eventos, el mismo usado desde la sesion 08.

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql.functions import col, from_json, current_timestamp, unix_millis

schema_evento = StructType([
    StructField("tipoEvento", StringType(), True),
    StructField("ordenId", LongType(), True),
    StructField("total", DoubleType(), True),
    StructField("estado", StringType(), True),
    StructField("origen", StringType(), True),
    StructField("timestamp", LongType(), True),
])

# Esta celda solo define el stream; no ejecuta nada hasta usar writeStream.
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", TOPIC_ORDENES) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

df_stream_events = df_kafka.select(
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("kafkaTimestamp"),
    col("value").cast("string").alias("value")
).select(
    "topic",
    "partition",
    "offset",
    "kafkaTimestamp",
    from_json(col("value"), schema_evento).alias("evento")
).select("topic", "partition", "offset", "kafkaTimestamp", "evento.*")

df_stream_observable = df_stream_events \
    .withColumn(
        "isValid",
        col("tipoEvento").isNotNull()
        & col("ordenId").isNotNull()
        & col("total").isNotNull()
        & col("timestamp").isNotNull()
    ) \
    .withColumn("processedAt", unix_millis(current_timestamp())) \
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))

df_stream_observable.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



## 11. Construir features streaming

En streaming no usamos `orderBy`, porque el flujo no tiene un orden global cerrado. En su lugar agregamos por ventanas con watermark y dejamos que Spark procese micro-batches.

In [10]:
serie_stream = df_stream_observable \
    .filter(F.col("isValid")) \
    .filter(F.col("kafkaTimestamp").isNotNull()) \
    .filter(F.col("latencyMs").isNotNull()) \
    .withWatermark("kafkaTimestamp", "2 minutes") \
    .groupBy(F.window("kafkaTimestamp", "1 minute")) \
    .agg(
        F.count("*").alias("eventCount"),
        F.avg("latencyMs").alias("avgLatencyMs"),
        F.min("latencyMs").alias("minLatencyMs"),
        F.max("latencyMs").alias("maxLatencyMs"),
        F.stddev("latencyMs").alias("stdLatencyMs")
    ) \
    .withColumn("windowStart", F.col("window.start")) \
    .withColumn("windowEnd", F.col("window.end")) \
    .drop("window") \
    .na.fill({"stdLatencyMs": 0.0})

features_stream = add_inference_features(serie_stream)
stream_predictions = loaded_model.transform(features_stream)

stream_predictions.printSchema()

root
 |-- eventCount: long (nullable = false)
 |-- avgLatencyMs: double (nullable = true)
 |-- minLatencyMs: long (nullable = true)
 |-- maxLatencyMs: long (nullable = true)
 |-- stdLatencyMs: double (nullable = false)
 |-- windowStart: timestamp (nullable = true)
 |-- windowEnd: timestamp (nullable = true)
 |-- latencyRangeMs: long (nullable = true)
 |-- hourOfDay: integer (nullable = true)
 |-- minuteOfHour: integer (nullable = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- prediction: double (nullable = false)



## 12. Salida streaming a consola

Esta consulta permite ver predicciones en vivo. Detenla cuando ya tengas evidencia suficiente.

In [11]:
def stop_query_if_exists(name):
    query = globals().get(name)
    if query is not None and query.isActive:
        query.stop()
        print(f"{name} detenida.")
    else:
        print(f"{name} no estaba activa.")


stop_query_if_exists("query_predictions_console")

query_predictions_console = stream_predictions.select(
    "windowStart",
    "windowEnd",
    "eventCount",
    F.round("avgLatencyMs", 2).alias("avgLatencyMs"),
    F.round("prediction", 2).alias("predictedNextAvgLatencyMs")
).writeStream \
    .queryName("ml_latency_predictions_console") \
    .format("console") \
    .outputMode("update") \
    .option("truncate", "false") \
    .start()

query_predictions_console

query_predictions_console no estaba activa.


26/06/08 03:39:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cebcc75a-2e4c-4373-8407-abcf437e3a50. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/08 03:39:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/08 03:39:02 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+---------+----------+------------+-------------------------+
|windowStart|windowEnd|eventCount|avgLatencyMs|predictedNextAvgLatencyMs|
+-----------+---------+----------+------------+-------------------------+
+-----------+---------+----------+------------+-------------------------+



26/06/08 04:07:28 WARN KafkaOffsetReaderAdmin: Error in attempt 1 getting Kafka offsets: 
java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.TimeoutException: Timed out waiting for a node assignment. Call: listOffsets(api=LIST_OFFSETS)
	at java.base/java.util.concurrent.CompletableFuture.reportGet(CompletableFuture.java:396)
	at java.base/java.util.concurrent.CompletableFuture.get(CompletableFuture.java:2073)
	at org.apache.kafka.common.internals.KafkaFutureImpl.get(KafkaFutureImpl.java:165)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.listOffsets(KafkaOffsetReaderAdmin.scala:87)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.$anonfun$fetchLatestOffsets$1(KafkaOffsetReaderAdmin.scala:337)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.$anonfun$partitionsAssignedToAdmin$1(KafkaOffsetReaderAdmin.scala:449)
	at org.apache.spark.sql.kafka010.KafkaOffsetReaderAdmin.withRetries(KafkaOffsetReaderAdmin.scala:466)
	at org.apache.spark.sql.kaf

## 13. Salida streaming a Parquet con foreachBatch

`foreachBatch` convierte cada micro-batch en un DataFrame batch normal. Esto permite guardar predicciones inferidas por el modelo en un destino reutilizable.

In [ ]:
def write_predictions_batch(microbatch_df, batch_id):
    rows = microbatch_df.count()
    if rows == 0:
        print(f"Microbatch {batch_id}: sin filas")
        return

    output_df = microbatch_df.select(
        F.lit(batch_id).alias("batchId"),
        "windowStart",
        "windowEnd",
        "eventCount",
        "avgLatencyMs",
        "minLatencyMs",
        "maxLatencyMs",
        "stdLatencyMs",
        "latencyRangeMs",
        "hourOfDay",
        "minuteOfHour",
        F.col("prediction").alias("predictedNextAvgLatencyMs")
    )

    output_df.write.mode("append").parquet(PREDICTIONS_STREAM_PATH)
    print(f"Microbatch {batch_id}: {rows} predicciones guardadas")


stop_query_if_exists("query_predictions_parquet")

query_predictions_parquet = stream_predictions.writeStream \
    .queryName("ml_latency_predictions_parquet") \
    .foreachBatch(write_predictions_batch) \
    .outputMode("update") \
    .option("checkpointLocation", CHECKPOINT_STREAM_PATH) \
    .start()

query_predictions_parquet

## 14. Leer predicciones streaming guardadas

Ejecuta esta celda despues de que hayan ingresado eventos y existan micro-batches guardados.

In [ ]:
if Path(PREDICTIONS_STREAM_PATH).exists():
    spark.read.parquet(PREDICTIONS_STREAM_PATH) \
        .orderBy(F.desc("batchId"), F.desc("windowStart")) \
        .show(20, truncate=False)
else:
    print("Todavia no existe salida streaming. Ejecuta la consulta y produce eventos en Kafka.")

## 15. Detener consultas streaming

In [ ]:
stop_query_if_exists("query_predictions_console")
stop_query_if_exists("query_predictions_parquet")

print("Queries activas:", len(spark.streams.active))

## 16. Que observar

- En **batch**, Spark lee ventanas historicas, aplica el `PipelineModel` y guarda resultados reproducibles.
- En **streaming**, Spark espera eventos, agrega por ventanas y aplica el mismo modelo dentro de cada micro-batch.
- El modelo no sabe si los datos vienen de archivos o de Kafka; solo necesita recibir las mismas columnas de features.
- `foreachBatch` es una frontera practica entre streaming y batch: permite persistir, auditar o enviar predicciones a otro sistema.

## 17. Cierre

Al terminar esta sesion debes tener:

- Modelo cargado desde `../artifacts/models/ml_latency_pipeline_lr`, creado en la sesion 09.
- Predicciones batch guardadas en `../artifacts/output/ml_latency_predictions_batch`.
- Ruta opcional para predicciones streaming en `../artifacts/output/ml_latency_predictions_stream`.

Con esto queda lista la base para la sesion 11: comparar modelos y seleccionar el mejor con validacion distribuida.

## Ejercicios sugeridos

1. Cambia la ventana de 1 minuto a 2 minutos y compara la estabilidad de las predicciones.
2. Agrega una regla de alerta cuando `predictedNextAvgLatencyMs > 250`.
3. Guarda tambien el error real cuando la siguiente ventana ya exista en batch.
4. Prueba el stream con `startingOffsets = earliest` para reprocesar eventos disponibles.